# Azure Well-Architected Framework

The Azure Well-Architected Framework (WAF) is a set of guiding principles for designing, building, and continuously improving cloud workloads on Azure. It organises architectural best practices into **five pillars**:

| Pillar | Core question |
|---|---|
| **Reliability** | Will the workload continue functioning correctly when things go wrong? |
| **Security** | Is the workload protected against threats and compliant with requirements? |
| **Cost Optimisation** | Are we getting maximum value from every dollar spent? |
| **Operational Excellence** | Can we deploy, monitor, and improve the workload confidently? |
| **Performance Efficiency** | Does the workload scale to meet demand without waste? |

The AWS equivalent is the AWS Well-Architected Framework — it uses the same five pillars with near-identical names, reflecting industry consensus on what good cloud architecture looks like.

## Well-Architected Review

Microsoft provides the **Azure Well-Architected Review** — a self-assessment questionnaire in the Azure portal (under Azure Advisor → Well-Architected) that evaluates a workload against the five pillars. Each question maps to a specific design principle. Answers generate prioritised recommendations with links to implementation guidance. The review is workload-scoped: you assess one specific application or service, not your entire Azure estate.

## Pillar 1 — Reliability

Reliability is the ability of a workload to recover from failures and continue to function. It encompasses two key metrics:

- **RTO (Recovery Time Objective)**: the maximum acceptable time for the workload to be unavailable after a failure
- **RPO (Recovery Point Objective)**: the maximum acceptable data loss, expressed as time (e.g. RPO = 1 hour means you can tolerate losing up to 1 hour of data)

### Design principles

**Design for failure** — assume every component will fail eventually. Use redundancy at every layer: multiple VM instances behind a load balancer, zone-redundant storage, geo-redundant databases. Remove all single points of failure.

**Use Availability Zones** — AZs are physically separate datacentres within a region with independent power, cooling, and networking. Deploying across three AZs (zone-redundant) tolerates the failure of an entire datacentre. Key zone-redundant services: Standard Load Balancer, Application Gateway v2, Azure SQL (zone-redundant option), Cosmos DB, AKS, VMSS.

**Use Availability Sets** — for IaaS VMs that cannot span AZs, Availability Sets distribute VMs across fault domains (separate physical racks) and update domains (separate maintenance windows). Use when AZ support is not available for a VM size.

**Health monitoring and self-healing** — use health probes on load balancers and application gateways to automatically remove unhealthy instances from rotation. Use VMSS health extensions to replace failed VMs automatically.

**Circuit breaker pattern** — when a downstream service is failing, stop sending requests to it immediately rather than queuing them. After a timeout, try a small number of probe requests. If they succeed, resume traffic. This prevents cascade failures where one slow service brings down the entire system.

**Retry with exponential backoff** — transient failures (network glitches, throttling) are normal in distributed systems. Retry failed operations with increasing delays (1s, 2s, 4s, 8s) and add jitter to avoid thundering herd problems. Azure SDKs implement this automatically.

**Chaos engineering** — intentionally inject failures in production (or staging) to validate that your redundancy actually works. Azure Chaos Studio provides managed fault injection experiments.

**Multi-region for high RTO/RPO requirements** — when a single region is insufficient, deploy active-passive (Traffic Manager failover) or active-active (Traffic Manager performance routing with Cosmos DB multi-region writes) across two or more regions.

## Pillar 2 — Security

Security in the WAF covers protecting workloads from threats and ensuring compliance. Microsoft's security model is built on **Zero Trust** principles:

> **Verify explicitly. Use least privilege. Assume breach.**

### Defence in depth

Security should be layered — an attacker who breaches one layer must still defeat all subsequent layers:

```
Physical security          (Microsoft's responsibility)
├── Identity & access      Entra ID, MFA, Conditional Access, PIM
├── Perimeter              DDoS Protection, Azure Firewall, WAF
├── Network                NSGs, Private Endpoints, VNet segmentation
├── Compute                Endpoint protection, JIT access, patching
├── Application            Secure coding, SAST, dependency scanning
└── Data                   Encryption at rest + in transit, Key Vault, BYOK
```

### Key security principles

**Identity is the new perimeter** — use Entra ID for all authentication. Require MFA for all users. Use Conditional Access policies to enforce MFA based on risk, location, and device compliance. Use Privileged Identity Management (PIM) for just-in-time elevation of privileged roles.

**Least privilege** — grant only the permissions required for a specific task and no more. Use Azure RBAC with built-in roles where possible. Avoid Owner and Contributor at the subscription level; assign roles at the resource group or resource scope.

**Eliminate secrets** — use Managed Identities for service-to-service authentication. Store all secrets in Key Vault. Use Workload Identity Federation for CI/CD pipelines. Never hardcode credentials in source code or configuration files.

**Network segmentation** — place workloads in separate subnets or VNets. Use NSGs to enforce allow-lists. Route traffic through Azure Firewall for inspection at the perimeter. Use Private Endpoints to keep service traffic off the public internet.

**Encrypt everything** — encryption at rest is automatic for all Azure storage and databases. Use customer-managed keys (CMK) via Key Vault for regulatory requirements. Enforce TLS 1.2+ for all data in transit. Use Azure Policy to deny non-HTTPS traffic.

**Assume breach** — enable Microsoft Defender for Cloud across all subscriptions. Enable Defender plans for the workload types you run. Export alerts to Microsoft Sentinel. Conduct regular penetration tests (permitted under the Microsoft Penetration Testing Rules of Engagement).

## Pillar 3 — Cost Optimisation

Cost Optimisation in the WAF is about getting maximum business value from your Azure spend — not simply minimising cost at the expense of reliability or performance.

### Design principles

**Choose the right consumption model** — serverless (Functions Consumption, Container Apps scale-to-zero) for unpredictable or low-traffic workloads. Reserved Instances for 24/7 steady-state compute. Spot VMs for batch and fault-tolerant workloads.

**Right-size continuously** — initial sizing is always a guess. Monitor actual utilisation and adjust. Azure Advisor Cost recommendations surface right-sizing opportunities automatically.

**Use managed services** — Azure SQL, Cosmos DB, App Service, and AKS are more expensive per unit than raw VMs, but they eliminate operational overhead. When you factor in the cost of patching, backups, HA configuration, and monitoring, managed services are often cheaper in total.

**Implement cost governance** — apply budgets at subscription and resource group level. Enforce resource tagging with Azure Policy so every resource has cost-centre, team, and environment tags. Use separate subscriptions for dev/test and production. Review Azure Advisor cost recommendations monthly.

**Architect for scale-down as well as scale-up** — auto-scaling should scale in aggressively when demand drops. A workload that scales out to 20 instances under load but scales back to 2 at night pays 10× less during off-peak hours.

**Eliminate waste proactively** — delete resources immediately after use in dev/test. Use lifecycle management on storage. Remove unattached disks, idle load balancers, and unused public IPs. Schedule auto-shutdown for non-production VMs.

## Pillar 4 — Operational Excellence

Operational Excellence is the ability to run and monitor the workload effectively and to continuously improve processes and procedures.

### Design principles

**Everything as code** — define all infrastructure as Bicep, Terraform, or ARM templates stored in source control. No manual portal changes in production. Infrastructure code is reviewed, tested, and deployed through the same CI/CD pipelines as application code.

**Automate deployments** — use Azure Pipelines or GitHub Actions for every deployment. Deployments should be repeatable, idempotent, and require no manual steps. Use blue-green or canary deployment strategies to reduce deployment risk.

**Observe everything** — instrument applications with Application Insights. Enable diagnostic logs on all resources and route to a Log Analytics workspace. Create dashboards and workbooks for operational visibility. Write KQL alerts for anomalous conditions.

**Define and track SLOs** — a Service Level Objective (SLO) is an internal target (e.g. 99.9% availability, p99 latency < 500ms). Track SLOs with Application Insights availability tests and custom metrics. An SLO breach is a signal to investigate and improve.

**Runbooks for common operations** — document every operational procedure (scaling, failover, key rotation, certificate renewal). Automate runbooks with Azure Automation or Azure Functions where possible. Regularly test runbooks so operators are familiar with them before an incident.

**Blameless post-mortems** — after every incident, conduct a structured review. Identify the root cause, contributing factors, and preventive actions. Focus on system improvements, not individual blame. Share post-mortems widely to spread learnings.

**Continuous improvement** — use the Well-Architected Review periodically (e.g. annually or after major architecture changes) to identify new improvement areas. Prioritise recommendations by potential impact and implement them iteratively.

**Safe deployment practices**:
- Feature flags to decouple deployment from feature release
- Deployment rings: canary → early adopters → production
- Automated rollback on error rate spike
- Deployment freeze windows during high-traffic periods

## Pillar 5 — Performance Efficiency

Performance Efficiency is the ability of the workload to scale to meet demand efficiently — neither under-provisioned (causing poor user experience) nor over-provisioned (wasting money).

### Design principles

**Scale horizontally, not vertically** — adding more instances (scale out) is more resilient and more cost-effective than using a single larger instance (scale up). Horizontal scaling removes single points of failure. Design stateless application tiers that can scale out freely.

**Use caching strategically**:
- **CDN / Front Door**: cache static assets and API responses at the edge — reduces latency globally and offloads origin
- **Azure Cache for Redis**: cache database query results, session state, and expensive computations in memory
- **HTTP caching headers**: `Cache-Control` and `ETag` let browsers and CDN nodes cache responses correctly

**Choose the right data store for each workload**:
- Relational queries with ACID transactions → Azure SQL Database
- Global distribution with flexible schema → Cosmos DB
- Time-series telemetry → Azure Data Explorer
- Session state and caching → Azure Cache for Redis
- Unstructured large objects → Blob Storage

**Async over sync for long operations** — replace synchronous HTTP calls for long-running operations with an async pattern: return 202 Accepted immediately, process in the background (Azure Functions, Service Bus), and notify via webhook or polling endpoint. This prevents client timeouts and improves perceived responsiveness.

**Load testing before production** — use Azure Load Testing (managed Apache JMeter service) to validate that the application meets performance targets under expected peak load. Identify bottlenecks before users do.

**Autoscaling based on meaningful metrics** — scale on application-level metrics (queue depth, request rate, active connections) rather than infrastructure metrics (CPU%). A web tier scaled on HTTP request rate responds faster to load changes than one scaled on CPU.

**Partition for scale** — databases and storage partition data to distribute load. Choose partition keys carefully: Cosmos DB partition key should distribute RU/s evenly across partitions (avoid hot partitions). Blob Storage uses the first part of the blob name to shard across storage nodes — randomise prefixes for high-throughput scenarios.

**Measure, don't guess** — use Application Insights to identify actual bottlenecks. The Application Map shows which service has the highest average latency. The Performance tab shows the slowest requests and the slowest dependency calls. Optimise the actual bottleneck, not what you assume it is.

## Architecture Patterns

Several cross-cutting patterns appear across multiple WAF pillars:

### Landing Zone
An Azure Landing Zone is a pre-configured, governance-ready subscription environment. It implements the management group hierarchy, Azure Policy baseline, network hub-and-spoke topology, centralised logging workspace, and identity foundation that all workloads share. The Azure Landing Zone Accelerator (Cloud Adoption Framework) provides reference implementations in Bicep and Terraform.

### Hub-and-Spoke networking
A central hub VNet hosts shared services (Azure Firewall, VPN/ExpressRoute gateway, Bastion, DNS Private Resolver). Spoke VNets host individual workloads and peer to the hub. All traffic between spokes and to the internet is routed through the hub firewall for inspection. This enforces consistent security policy without duplicating network appliances.

### Strangler Fig pattern
Incrementally migrate a monolith to microservices by routing specific functionality to new services behind the same API surface. The monolith is "strangled" gradually — functionality moves to Azure services piece by piece without a risky big-bang rewrite.

### CQRS (Command Query Responsibility Segregation)
Separate the read and write models. Writes go to a normalised transactional store (Azure SQL). Reads are served from a denormalised read model (Cosmos DB, Redis, Azure Search) optimised for query performance. Changes propagate from write to read store via event-driven updates (Event Grid, Service Bus).

### Event-driven architecture
Services communicate by publishing and subscribing to events rather than direct HTTP calls. Producers (Event Grid, Service Bus, Event Hubs) decouple senders from receivers, enabling independent scaling, loose coupling, and natural retry semantics.

## Code Demo — Well-Architected Tooling

In [ ]:
# pip install azure-mgmt-advisor azure-mgmt-resourcehealth azure-mgmt-monitor
# pip install azure-identity

from azure.identity import DefaultAzureCredential
from azure.mgmt.advisor import AdvisorManagementClient
import os

credential = DefaultAzureCredential()
subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]

advisor_client = AdvisorManagementClient(credential, subscription_id)

# --- List recommendations across all WAF pillars ---
all_recommendations = list(advisor_client.recommendations.list())

# Group by category (pillar)
from collections import defaultdict
by_category = defaultdict(list)
for rec in all_recommendations:
    category = rec.category or "Unknown"
    by_category[category].append(rec)

print("Azure Advisor recommendations by pillar:")
pillar_order = ["Cost", "Security", "Reliability", "OperationalExcellence", "Performance"]
for pillar in pillar_order:
    recs = by_category.get(pillar, [])
    high = sum(1 for r in recs if r.impact and r.impact.lower() == "high")
    medium = sum(1 for r in recs if r.impact and r.impact.lower() == "medium")
    low = sum(1 for r in recs if r.impact and r.impact.lower() == "low")
    print(f"  {pillar:<25} total={len(recs):>3}  high={high}  medium={medium}  low={low}")

In [ ]:
# --- List high-impact Reliability recommendations ---
reliability_high = [
    r for r in all_recommendations
    if r.category == "Reliability" and r.impact and r.impact.lower() == "high"
]

print(f"High-impact Reliability recommendations ({len(reliability_high)}):")
for rec in reliability_high[:5]:
    resource = rec.resource_metadata.resource_id.split("/")[-1] if rec.resource_metadata else "N/A"
    problem = rec.short_description.problem if rec.short_description else "N/A"
    solution = rec.short_description.solution if rec.short_description else "N/A"
    print(f"\n  Resource: {resource}")
    print(f"  Problem:  {problem}")
    print(f"  Solution: {solution}")

In [ ]:
# --- Resource Health: check availability of resources ---
from azure.mgmt.resourcehealth import MicrosoftResourceHealth

health_client = MicrosoftResourceHealth(credential, subscription_id)

# List current resource health events (service issues, planned maintenance)
print("Active service health events:")
try:
    events = health_client.events.list_by_subscription_id()
    event_list = list(events)
    if not event_list:
        print("  No active service health events — all services healthy.")
    for event in event_list[:5]:
        print(f"  [{event.event_type}] {event.title}")
        print(f"    Status: {event.status}  Level: {event.level}")
        print(f"    Services: {[s.impacted_service for s in (event.impact or [])[:3]]}")
except Exception as e:
    print(f"  (Could not retrieve events: {e})")

In [ ]:
# --- Chaos Studio: list experiments (fault injection for reliability testing) ---
# Azure Chaos Studio uses the Resource Management API
from azure.mgmt.resource import ResourceManagementClient

resource_client = ResourceManagementClient(credential, subscription_id)

# List Chaos Studio experiments if any exist
experiments = list(resource_client.resources.list(
    filter="resourceType eq 'Microsoft.Chaos/experiments'"
))

if experiments:
    print(f"Chaos Studio experiments ({len(experiments)}):")
    for exp in experiments:
        print(f"  {exp.name}  location={exp.location}")
else:
    print("No Chaos Studio experiments found.")
    print("To create one: Azure portal → Chaos Studio → Create experiment")
    print("Example faults: VM CPU pressure, AKS pod kill, SQL failover, network latency")

In [ ]:
# --- SLO tracking with Application Insights ---
# Calculate availability SLO for the last 7 days using the Log Analytics query API
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from datetime import timedelta

logs_client = LogsQueryClient(credential)

workspace_id = os.environ.get("LOG_ANALYTICS_WORKSPACE_ID", "<your-workspace-id>")

# Availability SLO: % of requests that succeeded in last 7 days
availability_query = """
requests
| where timestamp > ago(7d)
| summarize
    total = count(),
    successful = countif(success == true),
    failed = countif(success == false)
| extend availability_pct = round(todouble(successful) / todouble(total) * 100, 4)
| project total, successful, failed, availability_pct
"""

# p99 latency SLO
latency_query = """
requests
| where timestamp > ago(7d) and success == true
| summarize
    p50_ms  = percentile(duration, 50),
    p95_ms  = percentile(duration, 95),
    p99_ms  = percentile(duration, 99),
    p999_ms = percentile(duration, 99.9)
"""

print("SLO queries (run against your Application Insights workspace):")
print("\nAvailability SLO query:")
print(availability_query)
print("Latency SLO query:")
print(latency_query)

# Example of running the query (requires a real workspace ID)
if workspace_id != "<your-workspace-id>":
    result = logs_client.query_workspace(
        workspace_id=workspace_id,
        query=availability_query,
        timespan=timedelta(days=7)
    )
    if result.status == LogsQueryStatus.SUCCESS:
        for table in result.tables:
            for row in table.rows:
                print(f"Availability: {row['availability_pct']}% ({row['successful']}/{row['total']} requests)")

In [ ]:
# --- Azure Policy compliance summary (Operational Excellence governance) ---
from azure.mgmt.policyinsights import PolicyInsightsClient

policy_client = PolicyInsightsClient(credential)

# Summarise policy compliance for the subscription
summary = policy_client.policy_states.summarize_for_subscription(
    subscription_id=subscription_id
)

for s in summary.value:
    results = s.results
    if results:
        non_compliant_resources = results.non_compliant_resources or 0
        non_compliant_policies = results.non_compliant_policies or 0
        print(f"Policy compliance summary:")
        print(f"  Non-compliant resources: {non_compliant_resources}")
        print(f"  Non-compliant policies:  {non_compliant_policies}")

# List top non-compliant policy assignments
print("\nTop non-compliant policy assignments:")
for s in summary.value:
    for pa in (s.policy_assignments or [])[:5]:
        if pa.results and pa.results.non_compliant_resources:
            print(f"  {pa.policy_assignment_id.split('/')[-1]:<50} "
                  f"non-compliant={pa.results.non_compliant_resources}")

## Summary

| Pillar | Core principle | Key Azure services |
|---|---|---|
| **Reliability** | Design for failure; remove single points of failure | Availability Zones, Load Balancer, Traffic Manager, Chaos Studio, Azure SQL geo-replication |
| **Security** | Zero Trust: verify explicitly, least privilege, assume breach | Entra ID, Defender for Cloud, Key Vault, NSGs, Azure Firewall, Private Endpoints |
| **Cost Optimisation** | Match spend to business value; eliminate waste | Reserved Instances, Savings Plans, Spot VMs, Azure Advisor, Budgets, Tagging |
| **Operational Excellence** | Everything as code; automate; observe; improve | Azure Pipelines, Bicep/Terraform, Azure Monitor, Log Analytics, Application Insights |
| **Performance Efficiency** | Scale horizontally; cache; choose right data store | VMSS, AKS HPA, Azure CDN, Redis, Cosmos DB, Load Testing |

### Key cross-cutting recommendations

| Practice | Impact |
|---|---|
| Azure Landing Zone | Governance, networking, and identity foundation for all workloads |
| Hub-and-spoke networking | Centralised security inspection and consistent policy |
| Managed Identities everywhere | Eliminates credential management risk |
| Azure Policy for guardrails | Prevents misconfiguration at deployment time |
| Well-Architected Review | Periodic structured assessment to find improvement areas |
| Tagging + Budgets | Cost visibility and control at every scope |